# StatEO26 - Workshop Notebook: Execute, Analyse, and Visualise

This notebook is a **workshop guideline** for an end-to-end APEx scenario:

1. Discover an EO algorithm in the APEx Algorithm Catalogue.
2. Execute the service through an existing platform.
3. Publish outputs to an APEx Product Catalogue.
4. Visualise results in APEx Geospatial Explorer.

It is designed to be adapted live during the workshop.

## Pre-requisites

To get started with this notebook, please ensure that the following pre-requisites are met:

* You will need an account on CDSE to execute the openEO-based service from the APEx Algorithm Catalogue. Create your account by navigating to [this page](https://identity.dataspace.copernicus.eu/auth/realms/CDSE/account).

## Learning Goals

By the end of this session, participants should be able to:

- Navigate APEx Algorithm Catalogue metadata and identify a suitable service.
- Build and run an openEO workflow for EO-based statistics.
- Package and publish output metadata to a Product Catalogue.
- Configure a shareable map link for Geospatial Explorer-based communication.

In [32]:
# If needed, uncomment this line in a fresh environment:
# !pip install openeo pystac requests python-dotenv

import logging

logging.basicConfig(
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    level=logging.INFO,
)

## 1. Discover Candidate Algorithms (APEx Algorithm Catalogue Exploration)

This step demonstrates how to explore the existing services from the APEx Algorithm Catalogue. You can visually browse the APEx Algorithm Catalogue at https://algorithm-catalogue.apex.esa.int/. You can use the filtering methods to find the relevant service for the scenario. For example, you can view all statistics related algorithms at https://algorithm-catalogue.apex.esa.int/?labels=statistics.

/* SCREENSHOT OF CATALOGUE */

Once you have found your service, you can click on it to get the detailed information needed to execute the service. This information is available in the **Execution Information** tab on the page. 

/* SCREENSHOT OF DETAILS PAGE */

In [ ]:
process_id = "worldcover_statistics"
process = "https://raw.githubusercontent.com/ESA-APEx/apex_algorithms/refs/heads/main/algorithm_catalog/vito/worldcover_statistics/openeo_udp/worldcover_statistics.json"
backend = "https://openeo.dataspace.copernicus.eu"

## 2. Execute Workflow via openEO


### 2.1. Downloading Statistics
Now that we have our process selected, we can start executing it through openEO on the defined backend.

In [44]:
import openeo
from openeo.extra.job_management import MultiBackendJobManager

In [45]:
# Setting up the openEO Batch Job Manager

manager = MultiBackendJobManager()
manager.add_backend("backend", connection=openeo.connect(
    backend,
).authenticate_oidc())

2026-04-28 08:30:03,700 - openeo.rest.connection - INFO - Found OIDC providers: ['CDSE']
2026-04-28 08:30:03,703 - openeo.rest.connection - INFO - No OIDC provider given, but only one available: 'CDSE'. Using that one.
2026-04-28 08:30:03,966 - openeo.rest.connection - INFO - Using default client_id 'sh-a795b975-a105-40f8-80ff-1ca7220b7c3a' from OIDC provider 'CDSE' info.
2026-04-28 08:30:03,970 - openeo.rest.connection - INFO - Found refresh token: trying refresh token based authentication.
2026-04-28 08:30:03,970 - openeo.rest.auth.oidc - INFO - Doing 'refresh_token' token request 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token' with post data fields ['grant_type', 'client_id', 'refresh_token'] (client_id 'sh-a795b975-a105-40f8-80ff-1ca7220b7c3a')
2026-04-28 08:30:04,236 - openeo.rest.connection - INFO - Obtained tokens: ['token_type', 'access_token', 'expires_in', 'id_token', 'refresh_token', 'scope']
2026-04-28 08:30:04,240 - openeo.rest.aut

Authenticated using refresh token.


In [111]:
import pandas as pd
from openeo.extra.job_management import create_job_db

# Load AOI geometry from local GeoJSON file
with open('aoi.geojson', 'r', encoding='utf-8') as f:
    geometries = json.load(f)

df = pd.DataFrame({
    "geometry": geometries["features"],
    "feature_index": range(len(geometries["features"]))
})

stats_job_db = create_job_db("stateo_stats_jobs.csv", df=df,  on_exists="skip")
cogs_job_db = create_job_db("stateo_cogs_jobs.csv", df=df, on_exists="skip")

In [112]:
def start_stats_job(row, connection, **kwargs):
    statistics = connection.datacube_from_process(
        process_id=process_id,
        namespace=process,
        # Pass AOI geometry loaded from aoi.geojson
        geometries=row["geometry"],
    )
    return statistics.create_job(
        format="CSV",
        title=f"StatEO26 - {row['geometry']['properties']['name']} - APEx Workshop EO Statistics Demo Output",
        description='APEx workshop EO-based statistics demo output'
    )

def start_cog_job(row, connection, **kwargs):
    cog = connection.load_collection("ESA_WORLDCOVER_10M_2021_V2", spatial_extent=row["geometry"]).mask_polygon(row["geometry"])
    return cog.create_job(
        format="GTiff",
        title=f"StatEO26 - {row['geometry']['properties']['name']} - APEx Workshop EO COG Demo Output",
        description='APEx workshop EO-based COG demo output'
    )

In [113]:

# Start the job manager to calculate the statistics
manager.run_jobs(job_db=stats_job_db, start_job=start_stats_job)

2026-04-28 10:28:29,478 - openeo.extra.job_management._manager - INFO - Resuming `run_jobs` from existing CsvJobDatabase('stateo_stats_jobs.csv')
/Users/bramjanssen/miniconda/envs/apex_documentation/lib/python3.13/site-packages/openeo/extra/job_management/_job_db.py:131: FutureWarning: WKTReadingError is deprecated and will be removed in a future version. Use ShapelyError instead (functions previously raising {name} will now raise a ShapelyError instead).
  except shapely.errors.WKTReadingError:


defaultdict(int, {})

In [114]:
# Start the job manager to create COGs
manager.run_jobs(job_db=cogs_job_db, start_job=start_cog_job)

2026-04-28 10:28:33,603 - openeo.extra.job_management._manager - INFO - Resuming `run_jobs` from existing CsvJobDatabase('stateo_cogs_jobs.csv')
2026-04-28 10:28:33,616 - openeo.extra.job_management._manager - INFO - running_per_backend={} queued_per_backend={}
2026-04-28 10:28:33,621 - openeo.extra.job_management._manager - INFO - Starting job on backend backend for {'geometry': {'type': 'Feature', 'properties': {'name': 'Frascati'}, 'geometry': {'coordinates': [[[12.642238896420224, 41.8253194063314], [12.642238896420224, 41.7819287978451], [12.719336967510458, 41.7819287978451], [12.719336967510458, 41.8253194063314], [12.642238896420224, 41.8253194063314]]], 'type': 'Polygon'}}, 'feature_index': 0, 'id': None, 'backend_name': 'backend', 'status': 'not_started', 'start_time': None, 'running_start_time': None, 'cpu': None, 'memory': None, 'duration': None, 'costs': None}
2026-04-28 10:29:05,248 - urllib3.connectionpool - WARNING - Retrying (Retry(total=49, connect=None, read=49, redi

defaultdict(int,
            {'job_db persist': 8,
             'track_statuses': 5,
             'start_job call': 2,
             'job get status': 2,
             'job_queued_for_start': 2,
             'job launch': 2,
             'run_jobs loop': 5,
             'sleep': 5,
             'job describe': 6,
             'job start': 2,
             'job started running': 2,
             'job finished': 2,
             'job download': 2,
             'files downloaded': 2})

## 3. Post Processing

### 3.1. Creating the STAC items

Now that we have our results, we can start creating our STAC items. This is now a special case as we want to combine the outputs from the statistics with the geotiff into a single item with 2 assets.

In [256]:
from pathlib import Path
import csv
import json
from datetime import datetime

def _read_csv_rows(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))
    
def _find_matching_row(feature_index: dict, rows: list[dict]) -> dict | None:
        for row in rows:
            row_feature_index = row.get("feature_index")
            if row_feature_index not in (None, "") and str(row_feature_index) == str(feature_index):
                return row

def _load_job_stac(job_dir: Path) -> dict:
        candidates = [
            job_dir / "job-results.json",
        ]
        for candidate in candidates:
            if candidate.exists():
                with candidate.open("r", encoding="utf-8") as f:
                    return json.load(f)
        raise FileNotFoundError(f"No STAC metadata found in {job_dir}")

def _add_assets_from_job(item: pystac.Item, job_id: str, asset_prefix: str):
        job_dir = Path(f"job_{job_id}")
        stac_doc = _load_job_stac(job_dir)

        assets = stac_doc.get("assets", {})
        if not assets:
            print(f"Warning: no assets found in {job_dir}")
            return

        for asset_name, asset_meta in assets.items():
            href = asset_meta.get("href")
            if not href:
                continue

            item_asset = pystac.Asset.from_dict(asset_meta)
            item_asset.href = f"{job_dir}/{asset_name}"

            item.add_asset(
                key=f"{asset_prefix}_{asset_name}",
                asset=item_asset
            )

def _compute_bbox(geometry: dict) -> list[float]:
    coords = geometry.get("coordinates", [])[0]
    xs = [c[0] for c in coords]
    ys = [c[1] for c in coords]
    return [min(xs), min(ys), max(xs), max(ys)]


def _update_stats_json_mapping(job_id: dict, feature: list[dict]):
    stats_csv = Path(f"job_{job_id}/timeseries.csv")
    stats_json = Path(f"job_{job_id}/timeseries.json")

    label_mapping = {
        "treecover": "Tree cover",
        "shrubland": "Shrubland",
        "grassland": "Grassland",
        "cropland": "Cropland",
        "builtup": "Built-up",
        "bare/sparse vegetation": "Bare",
        "snow and ice": "Snow and ice",
        "water": "Permanent water bodies",
        "herbaceous wetland": "Herbaceous wetland",
        "mangroves": "Mangroves",
        "moss and lichen": "Moss and lichen",
    }
    
    # Read CSV row
    with stats_csv.open("r", encoding="utf-8", newline="") as f:
        row = list(csv.DictReader(f))[0]

    for key, value in row.items():
        if key == "feature_index":
            continue
        del feature["properties"][key]
        feature["properties"][label_mapping.get(key, key)] = value

    # Write merged output GeoJSON
    with stats_json.open("w", encoding="utf-8") as f:
        json.dump(feature, f, indent=2)
 

In [257]:
def create_stac_items_from_job_outputs(
    geometries: dict,
    stats_jobs_csv: str | Path = "stateo_stats_jobs.csv",
    cogs_jobs_csv: str | Path = "stateo_cogs_jobs.csv",
    jobs_root: str | Path = ".",
    asset_href_base: str | None = None,
):
    stats_rows = _read_csv_rows(Path(stats_jobs_csv))
    cog_rows = _read_csv_rows(Path(cogs_jobs_csv))
    jobs_root = Path(jobs_root)
    
    items = []
    for feature_index, feature in enumerate(geometries.get("features", [])):
        feature_props = dict(feature.get("properties", {}))
        item = pystac.Item(
            id=f"stateo26_demo_{feature_props.get('name', feature_index)}",
            geometry=feature["geometry"],
            bbox=_compute_bbox(feature["geometry"]),
            datetime=datetime.fromisoformat("2021-01-01T00:00:00Z"),
            properties=feature_props,
        )

        stats_row = _find_matching_row(feature_index, stats_rows)
        if stats_row and stats_row.get("id"):
            _add_assets_from_job(item, stats_row["id"], "stats")
            _update_stats_json_mapping(stats_row["id"], feature)
            item.properties["openeo_job_id"] = stats_row["id"]
        else:
            logging.warning(f"No matching stats job found for feature {feature_index} with properties {feature_props}")

        cog_row = _find_matching_row(feature_index, cog_rows)
        if cog_row and cog_row.get("id"):
            _add_assets_from_job(item, cog_row["id"], "cog")
            item.properties["openeo_job_id"] = cog_row["id"]
        else:
            logging.warning(f"No matching cog job found for feature {feature_index} with properties {feature_props}")

        items.append(item)

    return items


In [258]:
items = create_stac_items_from_job_outputs(
    geometries=geometries,
    stats_jobs_csv="stateo_stats_jobs.csv",
    cogs_jobs_csv="stateo_cogs_jobs.csv",
    jobs_root=".",
)

for item in items:
    print(json.dumps(item.to_dict(), indent=2))

{
  "type": "Feature",
  "stac_version": "1.1.0",
  "stac_extensions": [],
  "id": "stateo26_demo_Frascati",
  "geometry": {
    "coordinates": [
      [
        [
          12.642238896420224,
          41.8253194063314
        ],
        [
          12.642238896420224,
          41.7819287978451
        ],
        [
          12.719336967510458,
          41.7819287978451
        ],
        [
          12.719336967510458,
          41.8253194063314
        ],
        [
          12.642238896420224,
          41.8253194063314
        ]
      ]
    ],
    "type": "Polygon"
  },
  "bbox": [
    12.642238896420224,
    41.7819287978451,
    12.719336967510458,
    41.8253194063314
  ],
  "properties": {
    "name": "Frascati",
    "date": "2021-01-01T00:00:00.000Z",
    "treecover": "0.6525226954401618",
    "shrubland": "0.013352700108938112",
    "grassland": "0.12629558541266794",
    "cropland": "0.03577942625927271",
    "builtup": "0.17153083986097423",
    "bare/sparse vegetation"

### 3.2. Upload assets to APEx S3 bucket
Now that we have calculated the results, we see our first APEx tool in action. We can use the APEx S3 bucket to finally store our results.

In [259]:
import os

os.environ["APEX_S3_ACCESS_KEY_ID"] = "HPUAP2FLGTWGGPWCYEEP"
os.environ["APEX_S3_SECRET_ACCESS_KEY"] = "csgxTwHGJF3uofsgEeLh6XtGhHkia47PJreOYriK"

In [260]:
# If needed in a fresh environment:
# !pip install boto3

import os
from pathlib import Path
from urllib.parse import urlparse

import boto3
from botocore.config import Config

# Configure S3 access through environment variables
S3_CONFIG = {
    "endpoint_url": "https://obs.eu-nl.otc.t-systems.com",
    "region_name": "default",
    "bucket": "apex-prod-public",
    "prefix": "stateo26",
    "access_key": os.getenv("APEX_S3_ACCESS_KEY_ID"),
    "secret_key": os.getenv("APEX_S3_SECRET_ACCESS_KEY"),
}

# Some S3-compatible backends do not decode optional checksum/chunked framing correctly.
# Use strict settings so uploaded objects keep clean file content.
S3_CLIENT_CONFIG = Config(
    s3={
        "addressing_style": "virtual",
        "payload_signing_enabled": False,
    },
    request_checksum_calculation="when_required",
    response_checksum_validation="when_required",
)

def build_public_url(endpoint_url: str, bucket: str, key: str) -> str:
    parsed = urlparse(endpoint_url)
    return f"{parsed.scheme}://{bucket}.{parsed.netloc}/{key}"

In [261]:
# Create S3 client and upload generated assets (virtual-host style required by this endpoint)
s3_client = boto3.client(
    "s3",
    endpoint_url=S3_CONFIG["endpoint_url"],
    region_name=S3_CONFIG["region_name"],
    aws_access_key_id=S3_CONFIG["access_key"],
    aws_secret_access_key=S3_CONFIG["secret_key"],
    config=S3_CLIENT_CONFIG,
)

content_types = {
    ".json": "application/geo+json",
    ".csv": "text/csv",
    ".tif": "image/tiff",
}

upload_urls = []
for item in items:
    for asset_key, asset in item.assets.items():
        file_path = Path(asset.href)
        
        # Reuse the same object key each run; uploading to an existing key overwrites it.
        object_key = f"{S3_CONFIG['prefix'].strip('/')}/{item.properties.get('openeo_job_id', 'unknown')}/{file_path.name}".strip("/")

        # Use put_object with raw bytes to avoid chunked upload framing in object content.
        with file_path.open("rb") as f:
            body = f.read()

        s3_client.put_object(
            Bucket=S3_CONFIG["bucket"],
            Key=object_key,
            Body=body,
            ContentType=content_types.get(file_path.suffix.lower(), "application/octet-stream"),
            ACL="public-read",
        )
        asset.href = build_public_url(S3_CONFIG["endpoint_url"], S3_CONFIG["bucket"], object_key)

for item in items:
    print(json.dumps(item.to_dict(), indent=2))

{
  "type": "Feature",
  "stac_version": "1.1.0",
  "stac_extensions": [],
  "id": "stateo26_demo_Frascati",
  "geometry": {
    "coordinates": [
      [
        [
          12.642238896420224,
          41.8253194063314
        ],
        [
          12.642238896420224,
          41.7819287978451
        ],
        [
          12.719336967510458,
          41.7819287978451
        ],
        [
          12.719336967510458,
          41.8253194063314
        ],
        [
          12.642238896420224,
          41.8253194063314
        ]
      ]
    ],
    "type": "Polygon"
  },
  "bbox": [
    12.642238896420224,
    41.7819287978451,
    12.719336967510458,
    41.8253194063314
  ],
  "properties": {
    "name": "Frascati",
    "date": "2021-01-01T00:00:00.000Z",
    "treecover": "0.6525226954401618",
    "shrubland": "0.013352700108938112",
    "grassland": "0.12629558541266794",
    "cropland": "0.03577942625927271",
    "builtup": "0.17153083986097423",
    "bare/sparse vegetation"

## 4. Publish to APEx Product Catalogue

Next we can use the outputs from the openEO job to directly publish our results to an APEx Product Catalogue. 

In [262]:
stac_collection_id = f"apex-workshop-eo-stats-{datetime.now().strftime('%Y%m%d%H%M%S')}"

In [263]:
import pystac


temportal_extent = [datetime.fromisoformat("2021-01-01T00:00:00Z"), datetime.fromisoformat("2021-12-31T23:59:59Z")]

collection = pystac.Collection(
    id=stac_collection_id,
    title="APEx Workshop EO Statistics Demo",
    description="EO-based statistics collection from APEx workshop demo",
    extent=pystac.Extent(
        spatial=pystac.SpatialExtent([[-180, -90, 180, 90]]),
        temporal=pystac.TemporalExtent([temportal_extent])
    ),
    license = 'CC-BY-4.0',
)

collection.to_dict()

{'type': 'Collection',
 'id': 'apex-workshop-eo-stats-20260428154635',
 'stac_version': '1.1.0',
 'description': 'EO-based statistics collection from APEx workshop demo',
 'links': [],
 'title': 'APEx Workshop EO Statistics Demo',
 'extent': {'spatial': {'bbox': [[-180, -90, 180, 90]]},
  'temporal': {'interval': [['2021-01-01T00:00:00Z',
     '2021-12-31T23:59:59Z']]}},
 'license': 'CC-BY-4.0'}

First we need to obtain an access token for the APEx Catalogue API. This will be done by exchanging the client credentials, shared by the APEx team, with an actual token.

**Option A: set credentials in your terminal before starting Jupyter**

> These commands are intended for the terminal, not for a Python notebook cell.

```bash
export APEX_CLIENT_ID="your-client-id"
export APEX_CLIENT_SECRET="your-client-secret"
```

**Option B: set credentials for the current notebook session**

> Run the next cell and replace the placeholder values with your actual credentials.


In [264]:
os.environ["APEX_CLIENT_ID"] = "stateo26-catalogue-prod-m2m"
os.environ["APEX_CLIENT_SECRET"] = "rBHxrDwL1t4gqiginFJMQVz2Ua3del8b"

In [265]:
import os
import requests

apex_client_id = os.getenv('APEX_CLIENT_ID')
apex_client_secret = os.getenv('APEX_CLIENT_SECRET')

if not apex_client_id or not apex_client_secret:
    raise ValueError(
        'Set APEX_CLIENT_ID and APEX_CLIENT_SECRET first, either in your terminal or in the notebook setup cell above.'
    )

realm_url = 'https://auth.apex.esa.int/realms/apex'
token_url = f"{realm_url}/protocol/openid-connect/token"

token_resp = requests.post(
    token_url,
    data={
        'grant_type': 'client_credentials',
        'client_id': apex_client_id,
        'client_secret': apex_client_secret,
    },
    headers={'Content-Type': 'application/x-www-form-urlencoded'},
    timeout=30,
)
token_resp.raise_for_status()

token_payload = token_resp.json()
token = token_payload['access_token']

print('Token endpoint:', token_url)
print('Token acquired:', bool(token))

Token endpoint: https://auth.apex.esa.int/realms/apex/protocol/openid-connect/token
Token acquired: True


In [266]:
# Use the exchanged token above, or fall back to APEX_CATALOG_TOKEN from the environment
token = globals().get('token')

# Setting the Authorization header for the catalogue API
headers = {'Content-Type': 'application/json'}
if token:
    headers['Authorization'] = f'Bearer {token}'


In [267]:
# Setting the permissions for the collection, allowing anyone to read but only the current user to write
payload = collection.to_dict()
default_auth = {
    "_auth": {
        "read": ["anonymous"],
        "write": [os.getenv('APEX_CLIENT_ID')]
    }
}
payload.update(default_auth)
print(json.dumps(payload, indent=2))


{
  "type": "Collection",
  "id": "apex-workshop-eo-stats-20260428154635",
  "stac_version": "1.1.0",
  "description": "EO-based statistics collection from APEx workshop demo",
  "links": [],
  "title": "APEx Workshop EO Statistics Demo",
  "extent": {
    "spatial": {
      "bbox": [
        [
          -180,
          -90,
          180,
          90
        ]
      ]
    },
    "temporal": {
      "interval": [
        [
          "2021-01-01T00:00:00Z",
          "2021-12-31T23:59:59Z"
        ]
      ]
    }
  },
  "license": "CC-BY-4.0",
  "_auth": {
    "read": [
      "anonymous"
    ],
    "write": [
      "stateo26-catalogue-prod-m2m"
    ]
  }
}


In [268]:
# Publish the collection to the APEx catalogue (adjust URL if needed)
publish_resp = requests.post(
    'https://catalogue.stateo26.apex.esa.int/collections',
    headers=headers,
    data=json.dumps(payload),
    timeout=60
)
print('Publish status:', publish_resp.status_code)
print('Publish response:', publish_resp.text[:1000])


Publish status: 201
Publish response: {"id":"apex-workshop-eo-stats-20260428154635","description":"EO-based statistics collection from APEx workshop demo","stac_version":"1.1.0","links":[{"rel":"items","type":"application/geo+json","href":"https://catalogue.stateo26.apex.esa.int/collections/apex-workshop-eo-stats-20260428154635/items"},{"rel":"parent","type":"application/json","href":"https://catalogue.stateo26.apex.esa.int/"},{"rel":"root","type":"application/json","href":"https://catalogue.stateo26.apex.esa.int/"},{"rel":"self","type":"application/json","href":"https://catalogue.stateo26.apex.esa.int/collections"}],"title":"APEx Workshop EO Statistics Demo","type":"Collection","license":"CC-BY-4.0","extent":{"spatial":{"bbox":[[-180.0,-90.0,180.0,90.0]]},"temporal":{"interval":[["2021-01-01T00:00:00Z","2021-12-31T23:59:59Z"]]}},"_auth":{"read":["anonymous"],"write":["stateo26-catalogue-prod-m2m"]},"stac_extensions":[],"keywords":[],"providers":[],"summaries":{},"assets":{}}


In [269]:
# Post the items to the collection in the APEx catalogue
for item in items:
    item_payload = item.to_dict()
    item_payload["collection"]= stac_collection_id 
    print(json.dumps(item_payload, indent=2))
    post_resp = requests.post(
        f'https://catalogue.stateo26.apex.esa.int/collections/{stac_collection_id}/items',
        headers=headers,
        data=json.dumps(item_payload),
        timeout=60
    )
    print(f"Posting item {item.id} - status: {post_resp.status_code}, response: {post_resp.text[:1000]}")

{
  "type": "Feature",
  "stac_version": "1.1.0",
  "stac_extensions": [],
  "id": "stateo26_demo_Frascati",
  "geometry": {
    "coordinates": [
      [
        [
          12.642238896420224,
          41.8253194063314
        ],
        [
          12.642238896420224,
          41.7819287978451
        ],
        [
          12.719336967510458,
          41.7819287978451
        ],
        [
          12.719336967510458,
          41.8253194063314
        ],
        [
          12.642238896420224,
          41.8253194063314
        ]
      ]
    ],
    "type": "Polygon"
  },
  "bbox": [
    12.642238896420224,
    41.7819287978451,
    12.719336967510458,
    41.8253194063314
  ],
  "properties": {
    "name": "Frascati",
    "date": "2021-01-01T00:00:00.000Z",
    "treecover": "0.6525226954401618",
    "shrubland": "0.013352700108938112",
    "grassland": "0.12629558541266794",
    "cropland": "0.03577942625927271",
    "builtup": "0.17153083986097423",
    "bare/sparse vegetation"

The new collection is now available in the APEx Product Catalogue at https://browser.stateo26.apex.esa.int/

In [248]:
# Clean up by deleting the collection again (optional)
delete_resp = requests.delete(
    f"https://catalogue.stateo26.apex.esa.int/collections/apex-workshop-eo-stats-20260428111343",
    headers=headers,
    timeout=30
)
print('Delete status:', delete_resp.status_code)

Delete status: 204


## 5. Visualise in APEx Geospatial Explorer

Use the published collection/item URL or direct asset URL to build a shareable workshop link.
Exact query parameters may differ per Explorer deployment/version.

In [ ]:
# Example: link to records endpoint filtered by collection id
collection_query_url = f"{CFG['product_catalogue_records_url']}?q={quote(CFG['stac_collection_id'])}"

# Generic explorer handoff URL pattern (adapt to actual explorer route/params)
explorer_link = f"{CFG['geospatial_explorer_url']}?catalog={quote(collection_query_url, safe=':/?=&')}"

print('Collection query URL:')
print(collection_query_url)
print('\nProposed Explorer link:')
print(explorer_link)